<a href="https://colab.research.google.com/github/projects2026i-code/ai-automation-programme-portfolio/blob/main/agent_orchestration_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install openai langgraph -q

In [6]:
from google.colab import userdata
from openai import AzureOpenAI

endpoint = userdata.get('AZURE_ENDPOINT')
key = userdata.get('AZURE_KEY')
deployment = userdata.get('AZURE_DEPLOYMENT')

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2024-10-21"
)

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)

print(response.choices[0].message.content)

Hello!


In [7]:
import os
print(os.listdir('/content'))

['.config', 'sop.pdf', 'phase 2.pdf', 'meeting notes.pdf', 'pc.pdf', 'raid log.pdf', 'lessons.pdf', 'sample_data']


In [8]:
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 4.0 MB/s eta 0:00:00


In [9]:
from pypdf import PdfReader
import os

documents = {}

for filename in os.listdir('/content'):
    if filename.endswith('.pdf'):
        reader = PdfReader(f'/content/{filename}')
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        documents[filename] = text
        print(f"Loaded {filename}: {len(text)} characters")

print(f"\nTotal documents loaded: {len(documents)}")

Loaded sop.pdf: 8 characters
Loaded phase 2.pdf: 8 characters
Loaded meeting notes.pdf: 6 characters
Loaded pc.pdf: 7 characters
Loaded raid log.pdf: 6 characters
Loaded lessons.pdf: 10 characters

Total documents loaded: 6


In [10]:
for filename, text in documents.items():
    print(f"--- {filename} ---")
    print(repr(text))
    print()

--- sop.pdf ---
'\n\n\n\n\n\n\n\n'

--- phase 2.pdf ---
'\n\n\n\n\n\n\n\n'

--- meeting notes.pdf ---
'\n\n\n\n\n\n'

--- pc.pdf ---
'\n\n\n\n\n\n\n'

--- raid log.pdf ---
'\n\n\n\n\n\n'

--- lessons.pdf ---
'\n\n\n\n\n\n\n\n\n\n'



In [11]:
!apt-get install -y poppler-utils tesseract-ocr -q
!pip install pytesseract pdf2image -q

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 4 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.13 [186 kB]
Fetched 186 kB in 0s (1,172 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [12]:
from pdf2image import convert_from_path
import pytesseract
import os

documents = {}

for filename in os.listdir('/content'):
    if filename.endswith('.pdf'):
        pages = convert_from_path(f'/content/{filename}')
        text = ""
        for page in pages:
            text += pytesseract.image_to_string(page) + "\n"
        documents[filename] = text
        print(f"OCR'd {filename}: {len(text)} characters")

print(f"\nTotal documents loaded: {len(documents)}")

OCR'd sop.pdf: 10643 characters
OCR'd phase 2.pdf: 11988 characters
OCR'd meeting notes.pdf: 10059 characters
OCR'd pc.pdf: 10498 characters
OCR'd raid log.pdf: 6083 characters
OCR'd lessons.pdf: 16027 characters

Total documents loaded: 6


In [13]:
print(documents['sop.pdf'][:500])

STANDARD OPERATING PROCEDURE

Workflow Automation Platform — Daily Operations
Document ID: SOP-2022-AUTOMATION

Effective Date: 01-Feb-2022

Owner: Operations Manager

Last Reviewed: 31-Mar-2022

Version: 2.0

1. PURPOSE

This SOP defines the daily operational procedures for using the Workflow Automation Platform (WAP) to
process jobs, manage workflows, escalate exceptions, and track performance metrics in the vehicle
maintenance and logistics operations.

2. SCOPE

Applies to all operational st


In [14]:
!pip install scikit-learn numpy -q

In [15]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

all_chunks = []
chunk_sources = []

for filename, text in documents.items():
    chunks = chunk_text(text)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_sources.append(filename)

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk from {chunk_sources[0]}:")
print(all_chunks[0])

Total chunks created: 148
Example chunk from sop.pdf:
STANDARD OPERATING PROCEDURE

Workflow Automation Platform — Daily Operations
Document ID: SOP-2022-AUTOMATION

Effective Date: 01-Feb-2022

Owner: Operations Manager

Last Reviewed: 31-Mar-2022

Version: 2.0

1. PURPOSE

This SOP defines the daily operational procedures for using the Workflow Automation Platform (WAP) to
process jobs, manage workflows, escalate exceptions, and track performance metrics in the vehicle
maintenance and logistics operations.

2. SCOPE

Applies to all operational st


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words='english')
chunk_vectors = vectorizer.fit_transform(all_chunks)

print(f"Search index built: {chunk_vectors.shape[0]} chunks, {chunk_vectors.shape[1]} unique terms")

Search index built: 148 chunks, 1859 unique terms


In [17]:
def retrieve(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "text": all_chunks[idx],
            "source": chunk_sources[idx],
            "score": similarities[idx]
        })
    return results

# Test it
test_results = retrieve("What is the escalation process for exceptions?")
for r in test_results:
    print(f"[{r['source']}] score={r['score']:.3f}")
    print(r['text'][:200])
    print()

[phase 2.pdf] score=0.164
es

Competency assessment: 88% passing (target 85%)

7 staff below threshold; targeted coaching plan in place

Champions network expanded: 8 process champions + 12 peer mentors

Real-Time Agility Dash

[sop.pdf] score=0.135
parts delayed, etc.)
Operations Manager Dashboard

Live cycle time (current day)

Output (jobs completed per hour)
Exceptions (open + resolution time)
Utilisation (% of staff with jobs)

Quality metri

[sop.pdf] score=0.115
STANDARD OPERATING PROCEDURE

Workflow Automation Platform — Daily Operations
Document ID: SOP-2022-AUTOMATION

Effective Date: 01-Feb-2022

Owner: Operations Manager

Last Reviewed: 31-Mar-2022

Vers



In [18]:
def rag_answer(query, top_k=3):
    results = retrieve(query, top_k=top_k)

    context = "\n\n".join([
        f"[Source: {r['source']}]\n{r['text']}"
        for r in results
    ])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain the answer, say "I don't have enough information to answer that."
Always cite which source file(s) you used.

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content, results

# Test it
answer, sources = rag_answer("What is the escalation process for exceptions?")
print(answer)
print("\n--- Sources used ---")
for s in sources:
    print(f"- {s['source']} (score: {s['score']:.3f})")

I don't have enough information to answer that. (Sources: sop.pdf; phase 2.pdf)

--- Sources used ---
- phase 2.pdf (score: 0.164)
- sop.pdf (score: 0.135)
- sop.pdf (score: 0.115)


In [19]:
answer, sources = rag_answer("What is the purpose of the Workflow Automation Platform?")
print(answer)
print("\n--- Sources used ---")
for s in sources:
    print(f"- {s['source']} (score: {s['score']:.3f})")


The Workflow Automation Platform (WAP) is used to process jobs, manage workflows, escalate exceptions, and track performance metrics in the vehicle maintenance and logistics operations. (Source: sop.pdf)

--- Sources used ---
- sop.pdf (score: 0.408)
- phase 2.pdf (score: 0.214)
- phase 2.pdf (score: 0.112)


In [20]:
!pip install langgraph -q

In [21]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    query: str
    is_safe: bool
    retrieved: List[dict]
    grade: str
    rewritten_query: str
    answer: str
    attempts: int

def guardrail_node(state: AgentState):
    query = state["query"].lower()
    blocked_terms = ["ignore previous instructions", "system prompt", "jailbreak"]
    is_safe = not any(term in query for term in blocked_terms)
    return {"is_safe": is_safe, "attempts": 0}

def retrieve_node(state: AgentState):
    query = state.get("rewritten_query") or state["query"]
    results = retrieve(query, top_k=3)
    return {"retrieved": results}

def grade_node(state: AgentState):
    top_score = state["retrieved"][0]["score"] if state["retrieved"] else 0
    grade = "good" if top_score > 0.2 else "poor"
    return {"grade": grade}

def rewrite_node(state: AgentState):
    original = state["query"]
    prompt = f"""Rewrite this question as ONE single, more specific search query for a document retrieval system.
Return ONLY the rewritten query text, nothing else — no options, no explanation, no numbering.

Original question: '{original}'

Rewritten query:"""
    response = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "rewritten_query": response.choices[0].message.content.strip(),
        "attempts": state["attempts"] + 1
    }

def answer_node(state: AgentState):
    if not state["is_safe"]:
        return {"answer": "I can't help with that request."}
    answer, _ = rag_answer(state.get("rewritten_query") or state["query"])
    return {"answer": answer}

print("All nodes defined successfully")

All nodes defined successfully


In [22]:
def route_after_guardrail(state: AgentState):
    return "retrieve" if state["is_safe"] else "answer"

def route_after_grade(state: AgentState):
    if state["grade"] == "good" or state["attempts"] >= 2:
        return "answer"
    return "rewrite"

graph = StateGraph(AgentState)

graph.add_node("guardrail", guardrail_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("rewrite", rewrite_node)
graph.add_node("answer", answer_node)

graph.set_entry_point("guardrail")

graph.add_conditional_edges("guardrail", route_after_guardrail, {
    "retrieve": "retrieve",
    "answer": "answer"
})

graph.add_edge("retrieve", "grade")

graph.add_conditional_edges("grade", route_after_grade, {
    "answer": "answer",
    "rewrite": "rewrite"
})

graph.add_edge("rewrite", "retrieve")
graph.add_edge("answer", END)

agent = graph.compile()

print("Agent graph compiled successfully")

Agent graph compiled successfully


In [23]:
result = agent.invoke({
    "query": "What is the escalation process for exceptions?",
    "is_safe": True,
    "retrieved": [],
    "grade": "",
    "rewritten_query": "",
    "answer": "",
    "attempts": 0
})

print("FINAL ANSWER:")
print(result["answer"])
print(f"\nAttempts used: {result['attempts']}")
print(f"Final grade: {result['grade']}")
if result.get("rewritten_query"):
    print(f"Rewritten query used: {result['rewritten_query']}")

FINAL ANSWER:
I don't have enough information to answer that.

Source: sop.pdf

Attempts used: 2
Final grade: poor
Rewritten query used: Escalation process for exceptions step-by-step procedure roles and responsibilities escalation levels and triggers approval authorities SLAs and contact matrix


In [24]:
result = agent.invoke({
    "query": "What is the purpose of the Workflow Automation Platform?",
    "is_safe": True,
    "retrieved": [],
    "grade": "",
    "rewritten_query": "",
    "answer": "",
    "attempts": 0
})

print("FINAL ANSWER:")
print(result["answer"])
print(f"\nAttempts used: {result['attempts']}")
print(f"Final grade: {result['grade']}")

FINAL ANSWER:
The purpose of the Workflow Automation Platform (WAP) is to process jobs, manage workflows, escalate exceptions, and track performance metrics for vehicle maintenance and logistics operations (sop.pdf).

Attempts used: 0
Final grade: good


In [25]:
!pip install azure-storage-queue -q

In [26]:
from azure.storage.queue import QueueClient
from google.colab import userdata

queue_connection = userdata.get('AZURE_QUEUE_CONNECTION')
queue_name = "claims-intake-queue"

queue_client = QueueClient.from_connection_string(queue_connection, queue_name)

print("Connected to queue:", queue_name)

Connected to queue: claims-intake-queue


In [27]:
import json
import uuid

def submit_claim_event(claim_text):
    event = {
        "event_id": str(uuid.uuid4()),
        "event_type": "claim_submitted",
        "query": claim_text
    }
    queue_client.send_message(json.dumps(event))
    print(f"Event queued: {event['event_id']}")
    return event["event_id"]

# Test: simulate a new claim coming in
event_id = submit_claim_event("What is the escalation process for exceptions?")

Event queued: 58d607dd-15f4-4955-b70d-f59ee181fd11


In [28]:
def process_queue():
    messages = queue_client.receive_messages(max_messages=10)

    for msg in messages:
        event = json.loads(msg.content)
        print(f"Processing event {event['event_id']}: {event['query']}")

        result = agent.invoke({
            "query": event["query"],
            "is_safe": True,
            "retrieved": [],
            "grade": "",
            "rewritten_query": "",
            "answer": "",
            "attempts": 0
        })

        print(f"Answer: {result['answer']}")
        print(f"Attempts: {result['attempts']}, Grade: {result['grade']}\n")

        queue_client.delete_message(msg)

process_queue()

Processing event 22dfdb73-b2c6-4b1e-95b0-51cd14cf3fbf: What is the escalation process for exceptions?
Answer: I don't have enough information to answer that.

Available details in the provided context:
- Reviewers: Operations Manager, Finance Lead, IT; Document owner: Operations Manager (sop.pdf).
- Contact for questions: wap-support@company.com or ext. 5555; for staff confusion or system issues, contact Operations (sop.pdf).
- Role-based access and user roles referenced (Research Analyst, Compliance, Manager) (meeting notes.pdf).

Sources: meeting notes.pdf; sop.pdf
Attempts: 2, Grade: poor

Processing event 58d607dd-15f4-4955-b70d-f59ee181fd11: What is the escalation process for exceptions?
Answer: Partial answer from the provided policy (sop.pdf):

What the policy specifies
- Escalation channel: Escalations are performed via the WAP system (4.5 — Exception Handling & Escalation). [sop.pdf]
- Step 1 — Identify exception: Examples listed to trigger escalation: parts delayed; technical